<a href="https://colab.research.google.com/github/Annsujitra001/Lab-2/blob/main/Lab_2_Spatial_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-annsujitra001')
Map = geemap.Map()

# 1. ROI
roi = ee.FeatureCollection("FAO/GAUL/2015/level1") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Chiang Mai'))

print(roi.size().getInfo())
Map.centerObject(roi, 8)
Map.addLayer(roi, {}, "ROI")

# 2. Load data
collection = ee.ImageCollection("COPERNICUS/S2_SR") \
    .filterBounds(roi) \
    .filterDate('2023-01-01', '2023-02-15') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .select(['B2','B3','B4','B8'])

image = collection.median().clip(roi)

# NDVI ช่วง Jan
ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# แสดง Jan
Map.addLayer(ndvi, {'min': -1, 'max': 1, 'palette': ['blue','white','green']}, 'NDVI (Jan 2023)')

# ใส่ April
collection2 = ee.ImageCollection("COPERNICUS/S2_SR") \
    .filterBounds(roi) \
    .filterDate('2023-04-01', '2023-05-15') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .select(['B2','B3','B4','B8'])

image2 = collection2.median().clip(roi)

ndvi2 = image2.normalizedDifference(['B8', 'B4']).rename('NDVI')

Map.addLayer(ndvi2, {'min': -1, 'max': 1, 'palette': ['blue','white','green']}, 'NDVI (Apr 2023)')

# Zonal Statistics (ระดับอำเภอ)
admin = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Chiang Mai'))

zonal = ndvi.reduceRegions(
    collection=admin,
    reducer=ee.Reducer.mean(),
    scale=30
)

print(zonal.limit(5).getInfo())

# คำนวณการเปลี่ยนแปลง
# ทำ mask ให้ตรงกัน
mask = ndvi.mask().And(ndvi2.mask())

ndvi_diff = ndvi2.subtract(ndvi).updateMask(mask).clip(roi)

# สี
diff_vis = {
    'min': -0.3,
    'max': 0.3,
    'palette': ['darkred', 'white', 'darkgreen']
}

# แสดงผล
Map.addLayer(ndvi_diff, diff_vis, 'NDVI Change')

# 3. Index
ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

# 4. Visualization
ndvi_vis = {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'green']}
ndwi_vis = {'min': -1, 'max': 1, 'palette': ['brown', 'white', 'blue']}

Map.addLayer(ndvi, ndvi_vis, 'NDVI')
Map.addLayer(ndwi, ndwi_vis, 'NDWI')

Map

1
{'type': 'FeatureCollection', 'columns': {'ADM0_CODE': 'Integer', 'ADM0_NAME': 'String', 'ADM1_CODE': 'Integer', 'ADM1_NAME': 'String', 'ADM2_CODE': 'Integer', 'ADM2_NAME': 'String', 'DISP_AREA': 'String', 'EXP2_YEAR': 'Integer', 'STATUS': 'String', 'STR2_YEAR': 'Integer', 'Shape_Area': 'Float', 'Shape_Leng': 'Float', 'mean': 'Float<-1.0, 1.0>', 'system:index': 'String'}, 'features': [{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[99.01785267546889, 19.791278590771004], [99.01786161313092, 19.79038231756621], [99.01792847699305, 19.789642065853176], [99.01804441383626, 19.789062422988657], [99.01838775652438, 19.78825977260568], [99.01880690306874, 19.787813849141514], [99.01940885569252, 19.78743041614343], [99.0199974903596, 19.787194002465526], [99.02073773776587, 19.78705135507495], [99.02137537298971, 19.786975547073176], [99.02211107989389, 19.786881906227194], [99.02287360832017, 19.786792756107598], [99.02362720577973, 19.78667228346781], [99.0243227951

Map(center=[18.792236850995966, 98.73204090284445], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=ndvi.visualize(**ndvi_vis),
    description='NDVI_Jan2023',
    folder='GEE',
    fileNamePrefix='NDVI_Jan2023',
    region=roi.geometry(),
    scale=30,
    maxPixels=1e13
)

task.start()

In [ ]:
task2 = ee.batch.Export.image.toDrive(
    image=ndvi2.visualize(**ndvi_vis),
    description='NDVI_Apr2023',
    folder='GEE',
    fileNamePrefix='NDVI_Apr2023',
    region=roi.geometry(),
    scale=30,
    maxPixels=1e13
)

task2.start()

In [ ]:
task3 = ee.batch.Export.image.toDrive(
    image=ndvi_diff.visualize(**diff_vis),
    description='NDVI_Change',
    folder='GEE',
    fileNamePrefix='NDVI_Change',
    region=roi.geometry(),
    scale=30,
    maxPixels=1e13
)

task3.start()

In [ ]:
task_ndwi = ee.batch.Export.image.toDrive(
    image=ndwi.visualize(**ndwi_vis),
    description='NDWI',
    folder='GEE',
    fileNamePrefix='NDWI',
    region=roi.geometry(),
    scale=30,
    maxPixels=1e13
)

task_ndwi.start()